In [0]:
import os
from pathlib import Path
from pyspark.sql import functions as F
try:
    import kagglehub
except ModuleNotFoundError:
    !pip install kagglehub
    import kagglehub

# Funções:
Nessa seção serão setadas funções para auxiliar o estudo:

In [0]:
def get_tables_names_from_schema(table_schema:str)->list[str]:
    """
    This function returns all table names inside a table schema.
    parameters:
        table_schema(str): the schema of the table
    returns:
        table_names(list[str]): a list of table names
    """
    table_names = sorted([
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN mvp.{table_schema}").collect()
    ])
    return table_names


In [0]:
def show_table_full_schema(table_schema:str, table_name:str)->None:
    """
    This function shows the schema of a table in the catalog and schema.
    parameters:
        table_schema(str): the schema of the table
        table_name(str): the name of the table
    """
    print(f"SCHEMA FOR MVP.{table_schema.upper()}.{table_name.upper()}")
    spark.sql(f"""
        SELECT table_catalog, table_schema, table_name, comment
        FROM mvp.information_schema.tables
        WHERE table_schema = '{table_schema}'
        AND table_name = '{table_name}'
            """
    ).show(truncate=False)
    spark.sql(f"""
        SELECT column_name, data_type, comment
        FROM mvp.information_schema.columns
        WHERE table_schema = '{table_schema}'
        AND table_name = '{table_name}'
        """
    ).show(truncate=False)

# Perguntas a serem respondidas:
1. Qual categoria gera mais faturamento?
2. Produtos com mais fotos vendem mais?
3. Produtos entregues com atrazo impactam na avaliação?
4. Quais estados possuem maior volume de venda? 


# Download Dataset from kaggle:
Para esse trabalho, foi escolhido o dataset olist_brazilian_ecommerce do kaggle.<br>
Esse dataset foi escolhido por conter multiplas tabelas relacionadas em um contexto de ecommerce brasileiro. Possibilitando uma análise mais complexa e um processo de pipeline mais robusto. 
- dataset name: Brazilian E-Commerce Public Dataset by Olist
- dataset link: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
- dataset licence: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [0]:
dataset_path = "./dataset"

In [0]:
path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce",
    output_dir=dataset_path,    
    )

print("Path to dataset files:", path)

In [0]:
print(os.listdir(dataset_path))

In [0]:
# Get all csv files in a list
file_paths = [
    os.path.join(dataset_path, file) for file in os.listdir(dataset_path) 
    if os.path.isfile(os.path.join(dataset_path, file)) 
    and str(file).endswith("csv")
]
file_paths

# Análise exploratória:
Para entender melhor o dataset, foi realizado uma análise exploratória utilizando o polas. Essa análise teve como principal objetivo: 
- Identificar  e compreender as colunas; 
- Identificar possíveis erros;
- Estimar chaves primarias;
- Marcar possiveis relações/chaves estrangeiras;
- Facilitar a criação de um esquema entidade relacionamento do dataset bruto.

Essa análise esta no arquivo ./exploratory_analysis.ipynb

## Esquema ER do dataset bruto:
Como mencionado acima, foi feito o esquema de entidade-relacionamento do dataset bruto utilizando o drawsql.app:<br>
`obs:` Na camada bronze, todos os tipos das colunas são strings, sendo que a diferenciação da tipagem ira ocorrer somente na camada silver.
![raw dataset ER schema](./img/raw_er_schema.jpg)

# Camada Bronze:
Nessa seção, iremos criar a camada bronze do mvp, realizaremos a ingestão dos datasets brutos e adicionaremos os comentários da tabela e das colunas.<br>
Vale comentar, que por serem os dados brutos, iremos fazer a ingestão de todas as colunas como strings, realizando a tipagem em futuras camadas.<br>
Tanto a criação da camada, quanto a ingestão e comentario das tabelas serão feitas de forma automatica no python, enquanto que a inserção dos comentarios das colunas será feito utilizando a ia do databricks, devido a quantidade de colunas e tabelas.<br>

obs. Para a tabela de review, o modo de leitura do csv teve que ter as opções de multiLinear=True e foi setado o quote e escape. O motivo disso foi porque ao ler o csv somente com o separador, ocorreu um deslocamento nas colunas dessa tabela. Esse deslocamento ocorreu devido à coluna review_comment_message, que possui quebra de linha e outros valores de string.

## Passo a passo:
1. Criação e uso do catalogo mvp e do esquema mvp.bronze;
2. Leitura de todos os datasets em uma lista (contendo um dicionario com nome da lista e lista), iterando pela lista de arquivos;
3. Criação de um dicionario com os nome da lista e os comentários de cada tabela;
4. Ingestão ingestão dos dados e adição de comentarios das tabelas iterando pela lista de datasets (do item 2) correlacionando com o dicionario de comentarios (item 3);
5. Adição dos comentários das colunas;

In [0]:
# Create catalog:
spark.sql("CREATE CATALOG IF NOT EXISTS mvp")
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp.bronze")

In [0]:
# use schema and catalog:
spark.sql("USE CATALOG mvp")
spark.sql ("USE SCHEMA bronze")

In [0]:
raw_tables = []
for file_path in file_paths:
    file_name = os.path.split(file_path)[-1].removesuffix(".csv")
    absolute_path = str(Path(file_path).absolute())
    # for review_column, we will use multiLine option to read the column correctly
    if "review" not in file_name:
        raw_table = spark.read.option("header", True).option("sep", ",").csv(absolute_path)
    else:
        raw_table = spark.read.option("header", True).option("sep", ",").option("multiLine", True).option("quote", '"').option("escape", '"').csv(absolute_path)
    raw_tables.append({file_name: raw_table})
    print(file_name)
    display(raw_table.limit(10))


In [0]:
# Create comments of each table:
comments = {
    "olist_customers_dataset": "Contains information about the customer, including customer identifiers and geolocation information, like ZIP code prefix, state and city.",
    "olist_geolocation_dataset": "Contains information about customer's geolocation data, including ZIP code prefix, latitude and longiture coordnates, state and city.",
    "olist_order_items_dataset": "Contains information about the products included in each order, including product seller and order identifiers, price and shipping information, like limit date and freight value.",
    "olist_order_payments_dataset": "Contains information about the payment methods used for each order, including order and payment identifiers, payment method and payment value.",
    "olist_order_reviews_dataset": "Contains information about the reviews left by customers for each order, including review and order identifiers, score, comment title, comment message, creation date and anser timestamp.",
    "olist_orders_dataset": "Contains information about the orders, including order and customer identifiers, order status, purchase timestamp, order approved timestamp and delivery dates information.",
    "olist_products_dataset": "Contains information about the product, including identifier, category, name lenght, description lenght, quantity of photos, weight in grams and lenght, heigh and width in centimeters.",
    "olist_sellers_dataset": "Contains information about the sellers, including identifier, and geolocation information, like ZIP code prefix,state and city.",
    "product_category_name_translation": "Contains information about the product category name translation, including category name and translated name."
}

In [0]:
# Upload tabels in bronze schema and add a comment to each table:
for t in raw_tables:
    name, df = list(t.items())[0]

    # Remove _dataset from table name    
    table_name = name.removesuffix("_dataset") if name.endswith("_dataset") else name
    
    # Get comment
    comment = comments.get(name)
    
    # Create table
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

    # Add comment
    # Escape single quotes in comment by remove them
    escaped_comment = comment.replace("'", "") if comment else ""
    spark.sql(f"""
              COMMENT ON TABLE mvp.bronze.{table_name} IS 
              '{escaped_comment}'              
    """)


## Esquema das tabelas:
Devido a quantidade de tabelas e colunas nesse dataset, foi utilizado a ia do databricks para realizar os comentarios individuais das colunas de cada tabela. <br>
Isso foi feito da seguinte forma:
1. No menu de navegação, na esquerda, do databricks, abra a aba de catalogo;
2. Nela, vai ter um menu de navegação a direita do menu do databricks, contendo o nome do catalogo. abra o catalogo mvp;
3. Dentro dele tera o esquema bronze (ou outro esquema que você queira acessar, dependendo da seção do trabalho);
4. Dentro dele terão todas as tabelas criadas dentro desse esquema. apara cada tabela:
    1. Acesse a tabela desejada
    2. Na tela terá um esquema da tabela, na parte superior vai ter um botão na parte superior direita para usar a ia para detalhar essa tabela.
    3. Ao apertar no botão, revise os comentários e faça as alterações necessárias, como no caso das colunas com informação monetária, que foram adicionadas a informação referente á moeda utilizada (BRL).
    4. Após isso salve esses comentários.

`obs:` Esse passo a passo não se limita para a camada bronze, sendo assim, nas futuras camadas, ele não será reescrito.

In [0]:
bronze_table_names = get_tables_names_from_schema(table_schema="bronze")
bronze_table_names

In [0]:
for table_name in bronze_table_names:
    show_table_full_schema(
        table_name=table_name,
        table_schema="bronze"
    )

# Camada Prata
Nessa seção iremos:
1. Criar a camada prata do mvp;
2. Realizar a tipagem das colunas;
3. Renomear as colunas, caso necessário (o dataset olist adiciona o nome da tabela no inicio de cada coluna, sendo assim, removeremos o nome da tabela quando necessário);
3. Corrigir erros encontrados durante a análise exploratória;
4. Realizar a limpeza dos dados quando necessário (exclusão da tabela product_category_name_translation, que não será usada);

Nessa camada será realizado somente a exclusão da tabela com as traduções do category_name, não sendo excluído outras tabelas com informações não usadas na camada gold. Essa decisão foi tomada considerando que este trabalho se propõe a responder a apenas 4 perguntas de negócio. Dessa forma, a exclusão de outras tabelas ou colunas poderia resultar na perda de informações relevantes para a elaboração de novas análises e geração de insights em um contexto empresarial real.

In [0]:
# set catalog and create schema:
spark.sql("USE CATALOG mvp")
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp.silver")
spark.sql ("USE SCHEMA silver")

## Customers:
nessa tabela, todos as colunas são strings, sendo assim, não sera necessário realizar nenhuma tipagem ou correção.<br>
Portanto, somente iremos renomear as segunintes colunas:
- customer_zip_code_prefix -> zip_code_prefix; 
- customer_city -> city 
- customer_state -> state

In [0]:
table_name = "olist_customers"
table_name

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

In [0]:
# Rename columns:
df = (
    df
    .withColumnRenamed("customer_zip_code_prefix", "zip_code_prefix")
    .withColumnRenamed("customer_city", "city")
    .withColumnRenamed("customer_state", "state")
)
df.limit(10).display()

In [0]:
%sql
--check for errors in city:
SELECT *
FROM mvp.bronze.olist_customers
WHERE (
    try_cast(customer_city AS double) IS NOT NULL
)

In [0]:
# Save table in silver layer:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## GEOLOCATION:
Nessa tabela, iremos passar os valores de latitude e longitude para double e renomearemos as seguintes colunas:
- geolocation_zip_code_prefix -> zip_code_prefix
- geolocation_lat -> latitude
- geolocation_long -> longitude
- geolocation_city -> city
- geolocation_state -> state

In [0]:
table_name = "olist_geolocation"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

In [0]:
%sql
-- Check if all latitude values are numeric:
SELECT *
FROM mvp.bronze.olist_geolocation
WHERE (
    try_cast(geolocation_lat AS double) IS NULL OR
    try_cast(geolocation_lng AS double) IS NULL
)

In [0]:
# In the above query, we saw that all values of latitude and longitude are numeric
# Converting geolocation_lat and geolocation_lng to double
df = (
    df
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast("double"))
)
df.limit(10).display()

In [0]:
# Rename columns
df = (
    df
    .withColumnRenamed("geolocation_zip_code_prefix", "zip_code_prefix")
    .withColumnRenamed("geolocation_lat", "latitude")
    .withColumnRenamed("geolocation_lng", "longitude")
    .withColumnRenamed("geolocation_city", "city")
    .withColumnRenamed("geolocation_state", "state")
)
df.limit(10).display()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Order_items:
Nessa tabela iremos passar shipping_limit_date para timestamp e tanto price quanto freight_value para decimal(10,2).<br>
Após isso, como não há necessidade de renomear nenhuma das colunas, vamos salvá-la na camada silver.

In [0]:
table_name = "olist_order_items"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

In [0]:
%sql
-- Check if all price values can be casted as decimal:
SELECT *
FROM mvp.bronze.olist_order_items
WHERE (
    try_cast(price AS decimal(10,2)) IS NULL
    OR try_cast(freight_value AS decimal(10,2)) IS NULL
)

In [0]:
%sql
-- Check if shipping limit date can be casted as timestamp:
SELECT *
FROM mvp.bronze.olist_order_items
WHERE (
    try_cast(shipping_limit_date AS timestamp) IS NULL
)

In [0]:
# As seen in the queries above, all the prices columns can be converted as double(10,2)
# and the shipping_limit_date can be converted as timestamp.
df = (
    df
    .withColumn("price", F.col("price").cast("decimal(10,2)"))
    .withColumn("freight_value", F.col("freight_value").cast("decimal(10,2)"))
    .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast("timestamp"))
)
df.limit(10).display()

In [0]:
# Save table in silver layer:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Order_payments:
Nessa tabela vamos transformar a coluna payment_value para decimal(10, 2) e as colunas payment_sequential e payment_installments para inteiros.<br>
Por fim, como não é necessário renomear nenhuma coluna, vamos salvá-las direto na camada silver.

In [0]:
table_name = "olist_order_payments"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

In [0]:
%sql
-- Check if payment_sequential and payment_value can be casted as integer and the payment_value as double(10,2):
SELECT *
FROM mvp.bronze.olist_order_payments
WHERE (
    try_cast(payment_sequential AS int) IS NULL
    OR try_cast(payment_installments AS int) IS NULL
    OR try_cast(payment_value AS decimal(10,2)) IS NULL
)

In [0]:
# As all the target columns can be converted without lost of information.
df = (
    df
    .withColumn("payment_sequential", F.col("payment_sequential").cast("INT"))
    .withColumn("payment_installments", F.col("payment_installments").cast("INT"))
    .withColumn("payment_value", F.col("payment_value").cast("decimal(10,2)"))
)
df.limit(10).display()

In [0]:
# save table in silver layer
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Order_review:
Nessa tabela iremos converter as colunas review_creation_date e review_answer_timestamp para timestamp e review_score para int.<br> 
Além disso, iremos renomear as seguintes colunas:
- review_score -> score;
- review_comment_title -> title;
- review_comment_message -> message;
- review_creation_date -> creation_date;
- review_answer_timestamp -> answer_timestamp;

In [0]:
table_name = "olist_order_reviews"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

In [0]:
%sql
-- validate if we can convert review_score, review_creation_date and review_answer_timestamp without lose information
SELECT *
FROM mvp.bronze.olist_order_reviews
WHERE (
    try_cast(review_score AS int) IS NULL
)
LIMIT 10

In [0]:
%sql
-- validate if we can convert review_creation_date and review_answer_timestamp without lose information
SELECT *
FROM mvp.bronze.olist_order_reviews
WHERE (
    try_cast(review_creation_date AS timestamp) IS NULL
    OR (
        try_cast(review_answer_timestamp AS timestamp) IS NULL
        AND try_cast(review_answer_timestamp AS timestamp) IS NOT NULL
    )
)
LIMIT 10

In [0]:
# All target collumn can be converted without lose information:
df = (
    df
    .withColumn("review_score", F.col("review_score").cast("INT"))
.withColumn("review_creation_date", F.col("review_creation_date").cast("timestamp"))
.withColumn("review_answer_timestamp", F.col("review_answer_timestamp").cast("timestamp"))
)
df.limit(10).display()
            

In [0]:
# Rename columns
df = (
    df
    .withColumnRenamed("review_score", "score")
    .withColumnRenamed("review_comment_title", "title")
    .withColumnRenamed("review_comment_message", "message")
    .withColumnRenamed("review_creation_date", "creation_date")
    .withColumnRenamed("review_answer_timestamp", "answer_timestamp")
)
df.limit(10).display()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Order
Nessa tabela foi necessário converter as colunas order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date e order_estimated_delivery_date para timestamp. <br>
Além disso, as seguintes colunas foram renomeadas:
- order_status -> status
- order_purchase_timestamp -> purchase_timestamp
- order_delivered_customer_date -> delivered_customer_date
- order_delivered_carrier_date -> delivered_carrier_date
- order_estimated_delivery_date -> estimated_delivered_date

In [0]:
table_name = "olist_orders"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

In [0]:
%sql
-- validate if we can convert the columns whitout lose information:
SELECT *
FROM mvp.bronze.olist_orders
WHERE (
    try_cast(order_purchase_timestamp AS timestamp) IS NULL
    OR (
        try_cast(order_approved_at AS timestamp) IS NULL
        AND order_approved_at IS NOT NULL
        )
    OR (
        try_cast(order_delivered_carrier_date AS timestamp) IS NULL
        AND order_delivered_carrier_date IS NOT NULL
        )
    OR (
        try_cast(order_delivered_customer_date AS timestamp) IS NULL
        AND order_delivered_customer_date IS NOT NULL
        )
    OR (
        try_cast(order_estimated_delivery_date AS timestamp) IS NULL
        AND order_estimated_delivery_date IS NOT NULL
        )
)
LIMIT 10

In [0]:
# cast columns as timestamp:
df = (
    df
    .withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast("timestamp"))
    .withColumn("order_approved_at", F.col("order_approved_at").cast("timestamp"))
    .withColumn("order_delivered_carrier_date", F.col("order_delivered_carrier_date").cast("timestamp"))
    .withColumn("order_delivered_customer_date", F.col("order_delivered_customer_date").cast("timestamp"))
    .withColumn("order_estimated_delivery_date", F.col("order_estimated_delivery_date").cast("timestamp"))
)
df.limit(10).display()
    

In [0]:
# Rename columns:
df = (
    df
    .withColumnRenamed("order_status", "status")
    .withColumnRenamed("order_purchase_timestamp", "purchase_timestamp")
    .withColumnRenamed("order_delivered_customer_date", "delivered_customer_date")
    .withColumnRenamed("order_delivered_carrier_date", "delivered_carrier_date")
    .withColumnRenamed("order_estimated_delivery_date", "estimated_delivery_date")
)
df.limit(10).display()

In [0]:
# Save table in silver layer:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Products
Nessa tabela iremos transformar as colunas product_name_lenght, product_description_lenght e product_photos_qty para int e product_weight_g, product_lenght_cm, product_height_cm, product_widht_cm para decimal(8,2).<br>
Além disso, também serão renomeadas as seguintes colunas:
- product_category_name -> category_name;
- product_name_lenght -> name_lenght
- product_description_length -> description_length
- product_photos_qty -> photos_quantity
- product_weight_g -> weight_g
- product_length_cm -> length_cm
- product_height_cm -> height_cm
- product_width_cm -> width_cm

In [0]:
table_name = "olist_products"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

In [0]:
%sql
-- validate convertions:
SELECT * 
FROM mvp.bronze.olist_products
WHERE (
    (
        try_cast(product_name_lenght AS int) IS NULL
        AND product_name_lenght IS NOT NULL
    )
    OR ( 
        try_cast(product_description_lenght AS int) IS NULL
        AND product_description_lenght IS NOT NULL
        )
    OR (
        try_cast(product_photos_qty AS int) IS NULL
        AND product_photos_qty IS NOT NULL
        )
)

In [0]:
%sql
-- validate convertions:
SELECT * 
FROM mvp.bronze.olist_products
WHERE (
    try_cast(product_weight_g AS decimal(8, 2)) IS NULL
    OR try_cast(product_length_cm AS decimal(8, 2)) IS NULL
    OR try_cast(product_height_cm AS decimal(8, 2)) IS NULL
    OR try_cast(product_width_cm AS decimal(8, 2)) IS NULL
)

In [0]:
# change type of the columns:
df = (
    df
    .withColumn("product_weight_g", F.col("product_weight_g").cast("decimal(8,2)"))
    .withColumn("product_length_cm", F.col("product_length_cm").cast("decimal(8,2)"))
    .withColumn("product_height_cm", F.col("product_height_cm").cast("decimal(8,2)"))
    .withColumn("product_width_cm", F.col("product_width_cm").cast("decimal(8,2)"))
    .withColumn("product_name_lenght", F.col("product_name_lenght").cast("int"))
    .withColumn("product_description_lenght", F.col("product_description_lenght").cast("int"))
    .withColumn("product_photos_qty", F.col("product_photos_qty").cast("int"))
    )
df.limit(10).display()

In [0]:
# Rename Columns:
df = (
    df
    .withColumnRenamed("product_category_name", "category_name")
    .withColumnRenamed("product_name_lenght", "name_lenght")
    .withColumnRenamed("product_description_lenght", "description_lenght")
    .withColumnRenamed("product_photos_qty", "photos_quantity")
    .withColumnRenamed("product_weight_g", "weight_g")
    .withColumnRenamed("product_length_cm", "lengt_cm")
    .withColumnRenamed("product_height_cm", "height_cm")
    .withColumnRenamed("product_width_cm", "width_cm")
)


df.limit(10).display()

In [0]:
# Save table in silver layer
df = df.write.format("delta").mode("overwrite").saveAsTable(table_name)


## Sellers
Nessa tabela não faremos nenhuma transformação de tipo, mas iremos renomear as seguintes colunas:
- seller_zip_code_prefix -> zip_code_prefix;
- seller_city -> city;
- seller_state -> state;
Além disso, iremos corrigir um erro encontrado na tabela durante a análise exploratória.

In [0]:
table_name = "olist_sellers"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

### Correção de erro:
Durante a análise exploratória, foi identificado um erro dentro da tabela olist_sellers_dataset, mais especificamente na coluna sellers_city. Esse erro consiste no valor "04482255" no campo da cidade.<br>
Para corrigir esse erro, vamos utilizar a tabela de geolocalização, fazendo uma correlação de olist_sellers.sellers_zip_code com olistgeolocation.geolocation_zip_code. Com isso podemos pegar o valor correto da cidade da tabela de geolocalização e atualizar o valor incorreto da tabela sellers.

#### Passo a passo:
1. Mostrar o erro
2. Analisar a tabela de geolocalização para determinar que só existe um valor de cidade para o zip_code desejado
3. Caso exista somente uma cidade, subistituir esse valor pelo valor unico

In [0]:
%sql
-- Show error
SELECT * 
FROM mvp.bronze.olist_sellers
WHERE try_cast(seller_city AS int) IS NOT NULL

In [0]:
%sql
-- Verify if there is an unique value of seller city with the error
SELECT DISTINCT geolocation_zip_code_prefix, geolocation_city, geolocation_state
FROM mvp.bronze.olist_geolocation 
WHERE geolocation_zip_code_prefix = (
    SELECT seller_zip_code_prefix 
    FROM mvp.bronze.olist_sellers
    WHERE try_cast(seller_city AS int) IS NOT NULL
)

Como podemos ver acima, existe somente um valor de cidade para esse zip code. Com isso, vamos atualizar a tabela sellers para que seu valor seja 'rio de janeiro'

In [0]:
# Get unique correction for sellers table using geolocation table
geolocation_df = spark.read.table("mvp.silver.olist_geolocation")
sellers_city_correction = (
    geolocation_df
    .filter(geolocation_df.zip_code_prefix == "22790")
    .select("city")
    .distinct()
    .collect()
)
# Check if the value is unique
assert len(sellers_city_correction) == 1, f"geolocation_city contains {len(sellers_city_correction)} occurences for zip code prefix 22790"

# Get unique value
sellers_city_correction = sellers_city_correction[0]["city"]
sellers_city_correction

In [0]:
# Update sellers table:
df = df.withColumn(
    "seller_city",
    F.when(
        F.col("seller_city") == "04482255",
        sellers_city_correction
    ).otherwise(F.col("seller_city"))
)
# Assert that the change worked
assert (
    df
    .filter(df.seller_city == "04482255")
    .count() == 0
), "The correction did not work"

In [0]:
# Rename columns:
df = (
    df
    .withColumnRenamed("seller_zip_code_prefix", "zip_code_prefix")
    .withColumnRenamed("seller_city", "city")
    .withColumnRenamed("seller_state", "state")
)
df.limit(10).display()

In [0]:
# Save table in silver layer:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

Agora, para mostrar que realmente não temos mais esse erro:

In [0]:
%sql
SELECT * 
FROM mvp.silver.olist_sellers
WHERE city='04482255'

## Category Name Translation
Para esse trabalho, não iremos utilizar a tabela auxiliar de tradução do product_category_name. Sendo assim, iremos realizar uma verificação na tabela olist_product para ver se todos os valores da coluna product_category_name já estão em português. <br>
Caso a os valores já estiverem em portugês, nós não iremos passar a tabela olist_product_category_name_english para a camada silver.

In [0]:
%sql
--Check if all product_category_names are in portuguese
SELECT DISTINCT category_name 
FROM mvp.silver.olist_products
WHERE category_name IN (
    SELECT product_category_name_english
    FROM mvp.bronze.product_category_name_translation
)
AND category_name NOT IN (
    SELECT product_category_name
    FROM mvp.bronze.product_category_name_translation
)

Com base na busca acima, verificamos que todos os valores de product_category_name da tabela olist_product estão em portugês, sendo assim, não é necessário atualizar os valores da tabela de produtos.

## Validações finais:

### Verificando se todas as tabelas desejadas da camada bronze foram para a camada silver:

In [0]:
bronze_tables = get_tables_names_from_schema(table_schema="bronze")
silver_tables = get_tables_names_from_schema(table_schema="silver")

missing_tables = [t for t in silver_tables if t not in bronze_tables and t!="product_category_name_translation"]
assert len(missing_tables) == 0, f"Missing tables in silver layer: {missing_tables}"
print("All silver tables are valid")

### Esquema das tabelas:
Por fim, de forma similar à como foi feito na camada bronze, foi utilizado a ia do databricks para fazer os comentários de cada coluna e das tabelas criadas na camada silver. 

In [0]:
for table_name in silver_tables:
    show_table_full_schema(
        table_name=table_name,
        table_schema="silver",
    )

# Camada Ouro:
Para esse estudo, foi escolhido o modelo de flat_table, porquanto, no contexto das perguntas escolhidas, esse modelo simplifica o consumo e organização dos dados, reduzindo a necessidade de multiplos joins entre fatos e dimensões.<br>
Sendo assim, Para o contexto das perguntas a serem respondidas, serão feitas duas tabelas:
- sales_analysis:
    - order_id
    - order_item_id
    - product_id
    - category_name
    - photos_quantity
    - state
    - price

- order_review_analysis
    - order_id
    - customer_id
    - estimated_delivery_date
    - delivered_date
    - was_delayed
    - review_score


In [0]:
# Create catalog:
spark.sql("USE CATALOG mvp")
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp.gold")
spark.sql ("USE SCHEMA gold")

## Sales_analysis

In [0]:
%sql
-- Sales_analysis table will be created with olist_products, olist_orders_items and olist_customers information, and the olist_orders will be used to link olist_custom to olist_order_items.
CREATE OR REPLACE TABLE sales_analysis AS
SELECT order_items.order_id,
    order_items.order_item_id,
    order_items.product_id,
    product.category_name,
    product.photos_quantity,
    customers.state,
    order_items.price
FROM mvp.silver.olist_order_items order_items
INNER JOIN mvp.silver.olist_products product ON order_items.product_id = product.product_id
INNER JOIN mvp.silver.olist_orders orders ON order_items.order_id = orders.order_id
INNER JOIN mvp.silver.olist_customers customers ON orders.customer_id = customers.customer_id


In [0]:
%sql
-- Check sales_analysis table
SELECT * 
FROM mvp.gold.sales_analysis
LIMIT 10


In [0]:
%sql
-- Check if there are a null value in price
SELECT count(price) 
FROM sales_analysis
WHERE price IS NULL


## Order_Review_Analysis

In [0]:
%sql
-- order_review_analysis table will be created with mvp.silver.olist_orders e mvp.silver.olist_order_review, tendo uma coluna was_delay, que tera valor 1 para os casos aonde a data estimada foi menor que a data de entrega.
CREATE OR REPLACE TABLE order_review_analysis AS
SELECT orders.order_id,
    orders.customer_id,
    orders.estimated_delivery_date,
    orders.delivered_customer_date,
    CASE
        WHEN datediff(orders.delivered_customer_date, orders.estimated_delivery_date) > 0 THEN 1
        ELSE 0
    END AS was_delayed,
    review.score
FROM mvp.silver.olist_orders orders
INNER JOIN mvp.silver.olist_order_reviews review ON orders.order_id = review.order_id
WHERE (
    review.score IS NOT NULL
    AND orders.estimated_delivery_date IS NOT NULL
    AND orders.delivered_customer_date IS NOT NULL
    )


In [0]:
%sql
--check order_review_analysis table
SELECT * 
FROM order_review_analysis
LIMIT 10

In [0]:
%sql
--check order_review_analysis table
SELECT * 
FROM order_review_analysis
WHERE was_delayed = 1
LIMIT 10

In [0]:
%sql
--check order_review_analysis table
SELECT count(*) 
FROM order_review_analysis

In [0]:
%sql
--check order_review_analysis table
SELECT * 
FROM order_review_analysis
WHERE (
    estimated_delivery_date IS NULL
    OR delivered_customer_date IS NULL
    OR score IS NULL
)
LIMIT 10

## Esquema das tabelas:
De forma analoga a como foi feito nas camadas anteriores, foi usado a interface do Databricks para setar os comentarios das tabelas e das colunas faltantes (order_review_analysis.was_delayed), Isso foi feito utilizando a ia.<br>
O esquema das tabelas ficou da seguinte forma:

In [0]:
gold_table_names = get_tables_names_from_schema(table_schema="gold")
gold_table_names

In [0]:
for table_name in gold_table_names:
    show_table_full_schema(
        table_name=table_name,
        table_schema="gold"
    )

# Resposta das Perguntas:
Na seção abaixo, serão feitas as queryes para respnder as perguntas citadas no inicio desse trabalho.

In [0]:
spark.sql("USE CATALOG mvp")
spark.sql ("USE SCHEMA gold")

## 1. Qual categoria gera mais faturamento?
Para essa pergunta, será feita uma query simples agrupando o category_name pela soma do preço, organizado de forma decrescente. Dessa forma serão mostradas as categorias que tiveram maior faturamento (soma dos preços).<br>
De acordo com a query abaixo, a categoria que gera maior faturamento é **beleza_saude**, que gera R$:1.258.681,34. Seguido por relogios_presentes com faturamento de R$:1.205.005,68 e  cama_mesa_banho com faturamento de R$:1.036.988,68


In [0]:
%sql
SELECT category_name, sum(price) AS revenue
from sales_analysis
GROUP BY category_name
ORDER BY revenue DESC

## 2. Produtos com mais fotos vendem mais?
Para responder essa pergunta, será feita um agrupamento da coluna photo_quantity da tabela sales_analysis, usando um contador para contar quantas vendas tiveram para cada quantidade de fotos.<br>
De acordo com a query abaixo, **produtos com mais fotos tendem a ter menos vendas**. os produtos com mais volume de venda possuem 1 foto, tendo um volume de venda de 56028, seguido por produtos com 2 fotos, com 21963 vendas, 3 fotos com 12392 vendas e 4 fotos com 8437 vendas.<br>
Além disso, é interessante notar e produtos com nenhuma foto tiveram 1603 vendas, ficando em 7 lugar, enquanto que produtos com mais de 10 fotos ficaram a partir de 12º lugar.

In [0]:
%sql
SELECT photos_quantity, count(*) AS seles_volume
FROM sales_analysis
GROUP BY photos_quantity
ORDER BY seles_volume DESC

## 3. Produtos entregues com atrazo impactam na avaliação?
Para responder essa pergunta, será feito um agrupamento com o was_delayed, contendo o número de reviews, a média e o desvio padrão do score de revisão.<br>
De acordo com a query abaixo, **o atrazo na entrega de um produto possuem impacto na nota do review deixado**, sendo que produtos entregues em dia possuem uma nota média de 4.29 +- 1.15, enquanto que produtos entregues após o prazo estipulado tem uma média de 2.271 +- 1.571.

In [0]:
%sql

SELECT was_delayed,count(*) AS number_of_reviews, cast(mean(score) AS decimal(4, 3)) AS avg_score, cast(std(score) AS decimal(4, 3)) AS std_score
FROM order_review_analysis
GROUP BY was_delayed
ORDER BY was_delayed

## 4. Quais estados possuem maior volume de venda? 
Para responder essa pergunta, será feito um agrupamento dos estados da tabela sales_analysis, contendo um contador para o volume de vendas, mostrado em ordem decrescente.<br>
De acordo com a query abaixo, **o estado com maior volume de vendas foi São Paulo, com  47449 vendas**, seguido pelo Rio de Janeiro, com 14579 vendas e Minas Gerais com 13129 vendas.

In [0]:
%sql

SELECT state, count(*) AS sales_volume
FROM sales_analysis
GROUP BY state
ORDER BY sales_volume DESC

# Conclusão:
## Auto-Avaliação:
Esse trabalho conseguiu responder todas as perguntas formuladas no inicio de forma clara. Isso foi possivel devido às etapas de análise exploratória, que possibilitou a construção de um modelo de entidade-relacionamento, facilitando muito na etapa de construção do banco de dados e de tipagem durante a camada Silver.<br>

## Dificuldades:
Uma dificuldade encontrada durante esse trabalho foi na tabela de sales_review na camada bronze, porquanto a coluna com as mensagens das avaliações possuiam quebra de linha e outros caracteres, o que gerou um erro silencioso de deslocamento das colunas durante a leitura. Esse erro só foi encontrado durante a camada silver, quando foi realizado a tipagem das colunas e validação da perda de informação, porquanto tinham valores de review_id como timestamp. Para superar essa dificuldade, foi utilizado o notebook de análise exploratória, que mostrou que o erro estava na forma com que os dados estavam sendo lidos e não no dataset em sí. Com essa informação, foi realizado uma pesquisa no dataset junto com formas e opções de leitura e foi feito uma condicional, para ler somente essa tabela de forma diferente.<br>

## Futuros trabalhos:
Pensando nessa etapa, não foram excluidos as tabelas e informações não usadas na camada silver. Sendo assim, esse trabalho possue possibilidade de responder mas pergutas e gerar mais insights sobre esse dataset, por meio da criação de outras tabelas na camada gold. Além disso, para cada pergunta, podemos usar o pyspark junto com o pyplot/seaborn para gerar tabelas que melhorem a visualização desses dados, fazendo uma análise exploratória mais completa, e dos resultados, possibilitando uma melhor visualização das respostas.